Limpieza de archibo Restaurantes Busineess.

In [ ]:
#Librerias
import pandas as pd
import json
from textblob import TextBlob

In [ ]:
# Cargamos la ruta del archivo
input_file = '/content/drive/MyDrive/data/datos en bruto/Yelp/Yelps Restaurantes Busineess.json'
output_file = '/content/sample_data/Yelps_Restaurantes_Business_Limpio.json'

# Selecionamos las columnas que usaremos
columns = ['business_id', 'name', 'address', 'city', 'postal_code', 'latitude', 'longitude', 'stars', 'categories', 'attributes']

# Creamos una lista para almacenar los datos filtrados
filtered_data = []

# Leemos el archivo línea por línea
try:
    with open(input_file, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # Cargamos cada línea como un objeto JSON
                entry = json.loads(line.strip())
                # Filtramos  las columnas deseadas
                filtered_entry = {col: entry.get(col, None) for col in columns}
                filtered_data.append(filtered_entry)
            except Exception as e:
                print(f"Error procesando línea: {e}")
except Exception as e:
    print(f"Error al leer el archivo: {e}")

# Convertimos a DataFrame para limpieza adicional
df = pd.DataFrame(filtered_data)

# Manejamos los valores faltantes si es necesario
df.fillna({'address': '', 'postal_code': '', 'categories': '', 'attributes': {}}, inplace=True)

# Guardarmos el resultado
try:
    df.to_json(output_file, orient='records', lines=True, force_ascii=False)
    print(f"Archivo limpio guardado en: {output_file}")
except Exception as e:
    print(f"Error al guardar el archivo: {e}")


Archivo limpio guardado en: /content/sample_data/Yelps_Restaurantes_Business_Limpio.json


Limpimos el archivo fechas dejando solo el año

In [ ]:
# Cargamos la ruta del archivo de entrada y salida
input_file = '/content/drive/MyDrive/data/datos en bruto/Yelp/Yelpa fechas .json'
output_file = '/content/sample_data/Yelp_Fecha_Limpia.json'

# Creamos una  lista para almacenar los datos procesados
cleaned_data = []

# Leemos el archivo línea por línea
try:
    with open(input_file, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # Cargar cada línea como un objeto JSON
                entry = json.loads(line.strip())
                business_id = entry.get('business_id')
                dates = entry.get('date', '')

                # Extraer años de las fechas
                if dates:
                    years = list(set(date.split('-')[0] for date in dates.split(', ')))
                else:
                    years = []

                # Agregar datos limpios
                for year in years:
                    cleaned_data.append({'business_id': business_id, 'year': year})
            except Exception as e:
                print(f"Error procesando línea: {e}")
except Exception as e:
    print(f"Error al leer el archivo: {e}")

# Convertimos a DataFrame y guardamos en un  archivo
try:
    df = pd.DataFrame(cleaned_data)
    df.to_json(output_file, orient='records', lines=True, force_ascii=False)
    print(f"Archivo limpio guardado en: {output_file}")
except Exception as e:
    print(f"Error al guardar el archivo: {e}")


Archivo limpio guardado en: /content/sample_data/Yelp_Fecha_Limpia.json


Comparamos la columna id en ambos archibos para ber la coisidencia en bisineess_id

In [ ]:
# Función para leer JSON línea por línea
def leer_json_lineas(ruta_archivo):
    data = []
    with open(ruta_archivo, 'r') as archivo:
        for linea in archivo:
            try:
                data.append(json.loads(linea.strip()))
            except json.JSONDecodeError as e:
                print(f"Error decodificando línea: {linea.strip()} -> {e}")
    return data

# Cargamos las rutas de los archivos
archivo_fechas = '/content/sample_data/Yelp_Fecha_Limpia.json'
archivo_business = '/content/sample_data/Yelps_Restaurantes_Business_Limpio.json'

# Leemos los archivos
fechas_data = leer_json_lineas(archivo_fechas)
business_data = leer_json_lineas(archivo_business)

# Obtenemos las  listas de business_id
fechas_ids = {item['business_id'] for item in fechas_data}
business_ids = {item['business_id'] for item in business_data}

# Comparamos la columna business_id
coincidencias = fechas_ids.intersection(business_ids)
no_en_fechas = business_ids.difference(fechas_ids)
no_en_business = fechas_ids.difference(business_ids)

# Resultados
print(f"Total de coincidencias: {len(coincidencias)}")
print(f"IDs en Business pero no en Fechas: {len(no_en_fechas)}")
print(f"IDs en Fechas pero no en Business: {len(no_en_business)}")

print("Archivos de resultados guardados.")



Total de coincidencias: 8628
IDs en Business pero no en Fechas: 103
IDs en Fechas pero no en Business: 0
Archivos de resultados guardados.


Hacemos un join entre ambos archivos

In [ ]:
# Función para leer JSON línea por línea
def leer_json_lineas(ruta_archivo):
    data = []
    with open(ruta_archivo, 'r') as archivo:
        for linea in archivo:
            try:
                data.append(json.loads(linea.strip()))
            except json.JSONDecodeError as e:
                print(f"Error decodificando línea: {linea.strip()} -> {e}")
    return data

# Cargamos las rutas de los archivos
archivo_fechas = '/content/sample_data/Yelp_Fecha_Limpia.json'
archivo_business = '/content/sample_data/Yelps_Restaurantes_Business_Limpio.json'

# Leemos los archivos
fechas_data = leer_json_lineas(archivo_fechas)
business_data = leer_json_lineas(archivo_business)

# Creamos un diccionario para las fechas por business_id
fechas_dict = {item['business_id']: item.get('year', None) for item in fechas_data if 'business_id' in item}

# Realizamos el join
resultado = []
sin_fecha_count = 0
for business in business_data:
    business_id = business['business_id']
    year = fechas_dict.get(business_id, None)
    if year is None:
        sin_fecha_count += 1
    resultado.append({
        **business,
        'year': year
    })

# Ordenar por año
resultado.sort(key=lambda x: (x['year'] if x['year'] else '9999'))

# Guardamos el resultado en un archivo JSON
resultado_path = '/content/sample_data/restaurant yelp.json'
with open(resultado_path, 'w') as f:
    for item in resultado:
        f.write(json.dumps(item) + '\n')

print(f"Archivo combinado y ordenado guardado en: {resultado_path}")
print(f"Total de registros sin fecha: {sin_fecha_count}")

Archivo combinado y ordenado guardado en: /content/sample_data/restaurant yelp.json
Total de registros sin fecha: 103


Limpieza y analisis de datos de  Yelps_reviews y analisis de datos

In [ ]:
# Función para leer JSON línea por línea
def leer_json_lineas(ruta_archivo):
    data = []
    with open(ruta_archivo, 'r') as archivo:
        for linea in archivo:
            try:
                data.append(json.loads(linea.strip()))
            except json.JSONDecodeError as e:
                print(f"Error decodificando línea: {linea.strip()} -> {e}")
    return data

# Cargamos las rutas de los archivos
archivo_reviews = '/content/drive/MyDrive/data/datos en bruto/Yelp/Yelps_reviews.json'
archivo_reviews_limpio = '/content/sample_data/Yelps_reviews.json'

# Leemos el archivo
reviews_data = leer_json_lineas(archivo_reviews)

# Procesamos  los datos
resultado = []
for review in reviews_data:
    review_id = review.get('review_id')
    user_id = review.get('user_id')
    business_id = review.get('business_id')
    stars = review.get('stars')
    useful = review.get('useful')
    funny = review.get('funny')
    cool = review.get('cool')
    text = review.get('text', '')
    year = review.get('date', '')[:4]

    # Aplicamos el análisis de sentimientos ala columna text
    sentiment = TextBlob(text).sentiment.polarity
    if sentiment > 0:
        sentiment_score = 1
    elif sentiment < 0:
        sentiment_score = 0.5
    else:
        sentiment_score = 0

    # Guardamos los  resultado sin la columna  y eliminamosla columna text
    resultado.append({
        "review_id": review_id,
        "user_id": user_id,
        "business_id": business_id,
        "stars": stars,
        "useful": useful,
        "funny": funny,
        "cool": cool,
        "year": year,
        "sentiment": sentiment_score
    })

# Guardamos los resultados
with open(archivo_reviews_limpio, 'w') as f:
    for item in resultado:
        f.write(json.dumps(item) + '\n')

print(f"Archivo limpio guardado en: {archivo_reviews}")


Archivo limpio guardado en: /content/drive/MyDrive/data/datos en bruto/Yelp/Yelps_reviews.json


Eliminamos colunas imnesesarias y cabimos de nombre stars a puntuacion_usuarios

In [ ]:
# Cargamos el archivo JSON
file_path = '/content/sample_data/Yelps_reviews.json'
df = pd.read_json(file_path, lines=True)

# Eliminamos  las columnas 'useful', 'funny', 'cool'
df = df.drop(columns=['useful', 'funny', 'cool'])

# Renombramos  la columna 'stars' a 'puntuacion_usuarios'
df = df.rename(columns={'stars': 'puntuacion_usuarios'})

# Guardar el archivo modificado
output_path = '/content/sample_data/Yelps_reviews.json'
df.to_json(output_path, orient='records', lines=True)

print("Archivo procesado y guardado con éxito.")


Archivo procesado y guardado con éxito.


Unificamos los archivos

In [ ]:
# Cargar los datos desde los archivos JSON
business_file = "/content/sample_data/Business_Join_Fecha_Ordenado.json"
reviews_file = "/content/sample_data/Yelps_reviews.json"

# Leer los archivos JSON
business_df = pd.read_json(business_file, lines=True)
reviews_df = pd.read_json(reviews_file, lines=True)

# Realizar el join entre los dos DataFrames utilizando solo 'business_id' como clave
merged_df = pd.merge(business_df, reviews_df, on='business_id', how='inner')

# Mostrar los primeros registros del resultado
print(merged_df.head())

# Guardar el resultado del join en un archivo CSV
output_file = "/content/sample_data/RESTAURANT.csv"
merged_df.to_csv(output_file, index=False)
print(f"Archivo combinado guardado en: {output_file}")



              business_id                   name                   address  \
0  RQn7FdKEv4qEG3Q_Tua9Ew  Vincenzo's Ristorante  2454 N McMullen Booth Rd   
1  RQn7FdKEv4qEG3Q_Tua9Ew  Vincenzo's Ristorante  2454 N McMullen Booth Rd   
2  RQn7FdKEv4qEG3Q_Tua9Ew  Vincenzo's Ristorante  2454 N McMullen Booth Rd   
3  RQn7FdKEv4qEG3Q_Tua9Ew  Vincenzo's Ristorante  2454 N McMullen Booth Rd   
4  RQn7FdKEv4qEG3Q_Tua9Ew  Vincenzo's Ristorante  2454 N McMullen Booth Rd   

         city postal_code   latitude  longitude  stars  \
0  Clearwater       33759  28.011262 -82.707751    3.0   
1  Clearwater       33759  28.011262 -82.707751    3.0   
2  Clearwater       33759  28.011262 -82.707751    3.0   
3  Clearwater       33759  28.011262 -82.707751    3.0   
4  Clearwater       33759  28.011262 -82.707751    3.0   

                    categories  attributes  year_x               review_id  \
0  American (New), Restaurants         NaN  2010.0  lEuKAykYEaLPMUZf9bNvjw   
1  American (New), Restaur

Cambiamos de nombres las columnas

In [ ]:
#Cargamos los datos desde los archivos JSON
business_file = "/content/sample_data/Business_Join_Fecha_Ordenado.json"
reviews_file = "/content/sample_data/Yelps_reviews.json"

business_df = pd.read_json(business_file, lines=True)
reviews_df = pd.read_json(reviews_file, lines=True)

# Realizamos  el join entre los dos DataFrames utilizando solo 'business_id' como clave
merged_df = pd.merge(business_df, reviews_df, on='business_id', how='inner')

# Eliminamos las columnas poco relebantes
columns_to_drop = ["business_id", "attributes", "review_id", "user_id"]
merged_df = merged_df.drop(columns=columns_to_drop)

# Renombramos  las columnas a español para  una mejor comprención
columns_to_rename = {
    "name": "nombre",
    "address": "direccion",
    "city": "ciudad",
    "postal_code": "codigo_postal",
    "latitude": "latitud",
    "longitude": "longitud",
    "stars": "puntuacion_yelp",
    "year_x": "codina",
    "year_y": "s",
    "sentiment": "analisis_sentimientos",
    "puntuacion_usuarios": "puntuacion_usuarios"
}
merged_df = merged_df.rename(columns=columns_to_rename)

print(merged_df.head())

# Guardamos los resultados
output_file = "/content/sample_data/Cleaned_Merged_Business_Reviews.csv"
merged_df.to_csv(output_file, index=False)
print(f"Archivo combinado y procesado guardado en: {output_file}")


                  nombre                 direccion      ciudad codigo_postal  \
0  Vincenzo's Ristorante  2454 N McMullen Booth Rd  Clearwater         33759   
1  Vincenzo's Ristorante  2454 N McMullen Booth Rd  Clearwater         33759   
2  Vincenzo's Ristorante  2454 N McMullen Booth Rd  Clearwater         33759   
3  Vincenzo's Ristorante  2454 N McMullen Booth Rd  Clearwater         33759   
4  Vincenzo's Ristorante  2454 N McMullen Booth Rd  Clearwater         33759   

     latitud   longitud  puntuacion_yelp                   categories  codina  \
0  28.011262 -82.707751              3.0  American (New), Restaurants  2010.0   
1  28.011262 -82.707751              3.0  American (New), Restaurants  2010.0   
2  28.011262 -82.707751              3.0  American (New), Restaurants  2010.0   
3  28.011262 -82.707751              3.0  American (New), Restaurants  2010.0   
4  28.011262 -82.707751              3.0  American (New), Restaurants  2010.0   

   puntuacion_usuarios     s  an

Creamos las columnas id_nomres y id_ciudad

In [ ]:
# Cargamos  los datos desde los archivos CSV
merged_file = "/content/sample_data/Cleaned_Merged_Business_Reviews.csv"
ciudades_file = "/content/sample_data/ciudades_dim.csv"

merged_df = pd.read_csv(merged_file)
ciudades_df = pd.read_csv(ciudades_file)

# Eliminamos  la columna 'business_id'
merged_df = merged_df.drop(columns=["business_id"], errors='ignore')

# Creamos  una lista única de nombres
unique_names = merged_df["nombre"].unique()

# Creaamos un hash de 4 caracteres para cada nombre de restaurant
name_hashes = {
    name: hashlib.md5(name.encode()).hexdigest()[:4] for name in unique_names
}

# Añadimos  la columna 'id_nombre' al DataFrame original
merged_df["id_nombre"] = merged_df["nombre"].map(name_hashes)

# Creaamos un DataFrame para almacenar los nombres y sus hashes
names_df = pd.DataFrame({
    "id_nombre": list(name_hashes.values()),
    "nombre": list(name_hashes.keys())
})

#Mantenemos las id creada en el archivo ciudades_dim
ciudades_dict = dict(zip(ciudades_df["ciudad"], ciudades_df["id_ciudad"]))

# Identificamos si hay  ciudades nuevas de aver les damos una id
new_cities = merged_df["ciudad"].dropna().unique()
new_cities = [city for city in new_cities if city not in ciudades_dict]

# Creamos  nuevas IDs para las ciudades desconocidas
new_city_ids = {
    city: max(ciudades_dict.values(), default=0) + i + 1 for i, city in enumerate(new_cities)
}
ciudades_dict.update(new_city_ids)

# Actualizamos  el DataFrame de ciudades
new_ciudades_df = pd.DataFrame({
    "id_ciudad": list(new_city_ids.values()),
    "ciudad": list(new_city_ids.keys())
})
ciudades_df = pd.concat([ciudades_df, new_ciudades_df], ignore_index=True)

# Asignamos  los IDs de ciudad al DataFrame original
merged_df["id_ciudad"] = merged_df["ciudad"].map(ciudades_dict)

# Eliminamos la columna 'codina' y renombrar 's' a 'año'
merged_df = merged_df.drop(columns=["codina"], errors='ignore')
merged_df = merged_df.rename(columns={"s": "año"})

# Reordenamos las columnas
column_order = [
    "id_nombre", "nombre", "direccion", "id_ciudad", "ciudad", "codigo_postal",
    "latitud", "longitud", "puntuacion_yelp", "categories", "puntuacion_usuarios", "año", "analisis_sentimientos"
]
merged_df = merged_df[column_order]

# Ordenamos el DataFrame por 'id_nombre'
merged_df = merged_df.sort_values(by="id_nombre")

# Guardamos los archivos finales
merged_df.to_csv("/content/sample_data/Cleaned_Merged_Business_Reviews2.csv", index=False)
names_df.to_csv("/content/sample_data/nombres_restaurant.csv", index=False)
ciudades_df.to_csv("/content/sample_data/ciudades_dim.csv", index=False)

print("Archivos actualizados y guardados correctamente.")


Archivos actualizados y guardados correctamente.


De haber varios datos por año los unificamos y obtenemos el promedio por columna

In [ ]:
# Cargamos los datos desde los archivos CSV
merged_file = "/content/sample_data/Cleaned_Merged_Business_Reviews2.csv"
ciudades_file = "/content/sample_data/ciudades_dim.csv"
zipcodes_file = "/content/sample_data/zipcodes.csv"

# Leer los archivos CSV
merged_df = pd.read_csv(merged_file)
ciudades_df = pd.read_csv(ciudades_file)
zipcodes_df = pd.read_csv(zipcodes_file, delimiter=';')

# Agrupar por 'id_nombre' y 'año' y calcular los promedios
merged_df = (
    merged_df.groupby(['id_nombre', 'año'], as_index=False)
    .agg({
        'nombre': 'first',
        'direccion': 'first',
        'id_ciudad': 'first',
        'ciudad': 'first',
        'codigo_postal': 'first',
        'latitud': 'mean',
        'longitud': 'mean',
        'puntuacion_yelp': 'mean',
        'categories': 'first',
        'puntuacion_usuarios': 'mean',
        'analisis_sentimientos': 'mean'
    })
)

# Redondear columnas numéricas al entero más cercano o valores discretos
merged_df['puntuacion_yelp'] = merged_df['puntuacion_yelp'].round(1)
merged_df['puntuacion_usuarios'] = merged_df['puntuacion_usuarios'].round(1)
merged_df['analisis_sentimientos'] = merged_df['analisis_sentimientos'].round(1)
merged_df['analisis_sentimientos'] = merged_df['analisis_sentimientos'].apply(lambda x: 0 if x < 0.25 else (0.5 if x < 0.75 else 1))

# Asignar nuevos códigos postales basados en la ciudad
zipcode_map = zipcodes_df.set_index('City')['Zip Code'].to_dict()
merged_df['codigo_postal'] = merged_df['ciudad'].map(zipcode_map)

# Eliminar filas donde no se pudo asignar un código postal
merged_df = merged_df.dropna(subset=['codigo_postal'])

# Convertir los códigos postales a enteros
merged_df['codigo_postal'] = merged_df['codigo_postal'].astype(int)

# Reordenar las columnas
column_order = [
    "id_nombre", "nombre", "direccion", "id_ciudad", "ciudad", "codigo_postal",
    "latitud", "longitud", "puntuacion_yelp", "categories", "puntuacion_usuarios", "año", "analisis_sentimientos"
]
merged_df = merged_df[column_order]

# Guardar el archivo final
merged_df.to_csv("/content/sample_data/Cleaned_Merged_Business_Reviews1.csv", index=False)
print("Archivo actualizado y guardado correctamente.")



Archivo actualizado y guardado correctamente.


Clasificamos los datos por año

In [2]:
# Cargamos la ruta de entrada
input_file = "/content/Cleaned_Merged_Business_Reviews1(6).csv"

#Cargamos la ruta de salida
output_dir = "/content/drive/MyDrive/data/Datos Prosesados"

# Leemos el archivo de datos
df = pd.read_csv(input_file)

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Agrupamos por año y guardar archivos separados
for year, group in df.groupby('año'):

    csv_filename = f"{output_dir}/restaurantes_{year}.csv"
    parquet_filename = f"{output_dir}/restaurantes_{year}.parquet"

    # Guardarmos en formato CSV
    group.to_csv(csv_filename, index=False)

    # Guardamos en formato Parquet
    group.to_parquet(parquet_filename, index=False)

    print(f"Archivos para el año {year} guardados: {csv_filename}, {parquet_filename}")

print("Proceso completado con éxito.")


Archivos para el año 2005 guardados: /content/drive/MyDrive/data/Datos Prosesados/restaurantes_2005.csv, /content/drive/MyDrive/data/Datos Prosesados/restaurantes_2005.parquet
Archivos para el año 2006 guardados: /content/drive/MyDrive/data/Datos Prosesados/restaurantes_2006.csv, /content/drive/MyDrive/data/Datos Prosesados/restaurantes_2006.parquet
Archivos para el año 2007 guardados: /content/drive/MyDrive/data/Datos Prosesados/restaurantes_2007.csv, /content/drive/MyDrive/data/Datos Prosesados/restaurantes_2007.parquet
Archivos para el año 2008 guardados: /content/drive/MyDrive/data/Datos Prosesados/restaurantes_2008.csv, /content/drive/MyDrive/data/Datos Prosesados/restaurantes_2008.parquet
Archivos para el año 2009 guardados: /content/drive/MyDrive/data/Datos Prosesados/restaurantes_2009.csv, /content/drive/MyDrive/data/Datos Prosesados/restaurantes_2009.parquet
Archivos para el año 2010 guardados: /content/drive/MyDrive/data/Datos Prosesados/restaurantes_2010.csv, /content/drive/